# 🎭 Deepfake Detection — EfficientNet-B0 Training on Google Colab

**This notebook trains a deepfake detection model on a T4 GPU.**

### What this notebook does:
1. ✅ Mounts Google Drive & uploads your dataset
2. ✅ Installs dependencies
3. ✅ Trains EfficientNet-B0 with class-imbalance handling
4. ✅ Two-stage training (frozen backbone → full fine-tune)
5. ✅ Saves the best model to Google Drive
6. ✅ Downloads the model to your local machine

### Dataset Structure Expected:
```
archive/
├── train/
│   ├── real/   (*.jpg, *.png)
│   └── fake/   (*.jpg, *.png)
└── test/
    ├── real/
    └── fake/
```

> **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running!

## Step 1: 🔌 Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive mounted!')
print('Files in Drive:', os.listdir('/content/drive/MyDrive')[:10])

## Step 2: 📤 Upload Dataset

**Option A (Recommended): Upload ZIP from local machine**
- Zip your `archive` folder on your PC: right-click → Send to → Compressed folder → `archive.zip`
- Run the cell below to upload

**Option B: If already on Google Drive**
- Skip this cell and set `DATA_ROOT` in Step 3 to point to your Drive folder

In [ ]:
# OPTION A: Upload from your computer
from google.colab import files
import zipfile, shutil

print('Select your archive.zip file to upload...')
uploaded = files.upload()  # This opens a file picker

# Extract the zip
for fname in uploaded.keys():
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/dataset')
    os.remove(fname)

print('\nExtracted files:')
for root, dirs, files_list in os.walk('/content/dataset'):
    level = root.replace('/content/dataset', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:3]:
            print(f'{subindent}{f}')
        if len(files_list) > 3:
            print(f'{subindent}... ({len(files_list)} files total)')

In [ ]:
# OPTION B: Copy from Google Drive (if already uploaded there)
# Uncomment and edit the path below:

# DRIVE_DATASET_PATH = '/content/drive/MyDrive/archive'  # Change this!
# shutil.copytree(DRIVE_DATASET_PATH, '/content/dataset/archive')
# print('Copied from Drive!')

## Step 3: ⚙️ Configuration — SET YOUR PATHS HERE

In [ ]:
import os, glob

# ─── AUTO-DETECT dataset structure ───────────────────────────────────────────
def find_split_dirs(base):
    """Find train/ and test/ directories anywhere inside base."""
    candidates = {}
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            key = d.lower()
            if key in ('train', 'test', 'val') and key not in candidates:
                full = os.path.join(root, d)
                # verify it contains real/ or fake/
                subs = [s.lower() for s in os.listdir(full)]
                if 'real' in subs or 'fake' in subs:
                    candidates[key] = full
    return candidates

DATASET_BASE = '/content/dataset'   # Change if you used Option B
splits = find_split_dirs(DATASET_BASE)

print('Found splits:', splits)

TRAIN_ROOT = splits.get('train', '')
VAL_ROOT   = splits.get('test', splits.get('val', ''))

if not TRAIN_ROOT:
    # If no train/ folder, assume the base is train and use 20% as val
    TRAIN_ROOT = DATASET_BASE
    VAL_ROOT   = None
    print('⚠️  No train/ found — will split automatically.')

# Training hyperparameters
EPOCHS      = 30          # More epochs on GPU is fast!
BATCH_SIZE  = 32          # T4 can handle 32 easily
LR          = 1e-3        # Initial learning rate
IMAGE_SIZE  = 224
DROPOUT     = 0.4
SEED        = 42
CHECKPOINT  = '/content/drive/MyDrive/deepfake_best_model.pth'  # Saved to Drive!

print(f'\n📁 Train root : {TRAIN_ROOT}')
print(f'📁 Val root   : {VAL_ROOT}')
print(f'💾 Checkpoint : {CHECKPOINT}')

## Step 4: 📦 Install Dependencies

In [ ]:
!pip install -q timm facenet-pytorch scikit-learn matplotlib seaborn tqdm
print('✅ Dependencies installed!')

## Step 5: 🔍 Verify GPU & Dataset

In [ ]:
import torch

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU')

# Count dataset images
def count_images(root):
    if not root or not os.path.isdir(root):
        return 0, 0
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    real = sum(1 for f in os.listdir(os.path.join(root, 'real'))
               if Path(f).suffix.lower() in exts) if os.path.isdir(os.path.join(root, 'real')) else 0
    fake = sum(1 for f in os.listdir(os.path.join(root, 'fake'))
               if Path(f).suffix.lower() in exts) if os.path.isdir(os.path.join(root, 'fake')) else 0
    return real, fake

from pathlib import Path
tr_r, tr_f = count_images(TRAIN_ROOT)
vl_r, vl_f = count_images(VAL_ROOT) if VAL_ROOT else (0, 0)

print(f'\n📊 Dataset Summary:')
print(f'   Train: {tr_r} real + {tr_f} fake = {tr_r+tr_f} total')
print(f'   Val  : {vl_r} real + {vl_f} fake = {vl_r+vl_f} total')
if tr_f > 0:
    print(f'   Imbalance ratio (real/fake): {tr_r/tr_f:.2f}')

## Step 6: 🧠 Define Model, Dataset & Training

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from pathlib import Path
import numpy as np
import os, random, time
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ── Reproducibility ────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ── Dataset ────────────────────────────────────────────────────────────────
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

class DeepfakeDataset(Dataset):
    def __init__(self, root, transform=None, split_val=False, val_ratio=0.2):
        self.transform = transform
        self.samples = []  # (path, label)

        for label, class_name in [(0, 'real'), (1, 'fake')]:
            class_dir = os.path.join(root, class_name)
            if not os.path.isdir(class_dir):
                print(f'⚠️  Missing directory: {class_dir}')
                continue
            files = sorted([f for f in os.listdir(class_dir)
                           if Path(f).suffix.lower() in EXTENSIONS])
            self.samples += [(os.path.join(class_dir, f), label) for f in files]

        # Shuffle for splitting
        random.shuffle(self.samples)

        n_real = sum(1 for _, l in self.samples if l == 0)
        n_fake = sum(1 for _, l in self.samples if l == 1)
        print(f'  → Dataset "{os.path.basename(root)}": {n_real} real, {n_fake} fake ({len(self.samples)} total)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128,128,128))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

# ── Transforms ─────────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# ── Model ──────────────────────────────────────────────────────────────────
def create_model(dropout=0.4, pretrained=True):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, 1)
    )
    return model

# ── Sampler for class imbalance ─────────────────────────────────────────────
def make_sampler(dataset):
    labels = [s[1] for s in dataset.samples]
    counts = np.bincount(labels)
    weights = 1.0 / counts.astype(float)
    sample_weights = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

print('✅ Model, Dataset, Transforms defined!')

## Step 7: 📂 Create DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split

print('Loading datasets...')

train_ds_full = DeepfakeDataset(TRAIN_ROOT, transform=train_transform)

if VAL_ROOT and os.path.isdir(VAL_ROOT):
    val_ds = DeepfakeDataset(VAL_ROOT, transform=val_transform)
else:
    # Auto-split 80/20
    print('  ⚙️  Auto-splitting 80/20 train/val...')
    all_samples = train_ds_full.samples
    labels = [s[1] for s in all_samples]
    train_idx, val_idx = train_test_split(
        range(len(all_samples)), test_size=0.2,
        stratify=labels, random_state=SEED
    )
    # Create val dataset with val_transform
    val_ds = DeepfakeDataset(TRAIN_ROOT, transform=val_transform)
    val_ds.samples = [all_samples[i] for i in val_idx]
    train_ds_full.samples = [all_samples[i] for i in train_idx]
    print(f'  → Train: {len(train_ds_full)}, Val: {len(val_ds)}')

# Compute pos_weight for BCEWithLogitsLoss
train_labels = [s[1] for s in train_ds_full.samples]
n_real = sum(1 for l in train_labels if l == 0)
n_fake = sum(1 for l in train_labels if l == 1)
pos_weight = torch.tensor([n_real / max(n_fake, 1)], device=device)
print(f'\n⚖️  Class balance — Real: {n_real}, Fake: {n_fake}')
print(f'   pos_weight: {pos_weight.item():.3f}')

sampler = make_sampler(train_ds_full)

train_loader = DataLoader(
    train_ds_full, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=2,
    pin_memory=True, prefetch_factor=2
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=True
)

print(f'\n✅ DataLoaders ready!')
print(f'   Train batches: {len(train_loader)}')
print(f'   Val batches  : {len(val_loader)}')

## Step 8: 🚀 Training Loop

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings('ignore')

def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, all_probs, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc='Train', leave=False)

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        with autocast():
            logits = model(imgs).squeeze(1)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader.dataset)
    acc = ((np.array(all_probs) >= 0.5) == np.array(all_labels)).mean()
    return avg_loss, float(acc)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []

    for imgs, labels in tqdm(loader, desc='Val  ', leave=False):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with autocast():
            logits = model(imgs).squeeze(1)
            loss   = criterion(logits, labels)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total_loss += loss.item() * imgs.size(0)

    avg_loss   = total_loss / len(loader.dataset)
    probs_arr  = np.array(all_probs)
    labels_arr = np.array(all_labels)
    preds      = (probs_arr >= 0.5).astype(int)
    acc  = (preds == labels_arr).mean()
    try:   auc = roc_auc_score(labels_arr, probs_arr)
    except: auc = 0.0
    f1 = f1_score(labels_arr, preds, zero_division=0)
    return avg_loss, float(acc), float(auc), float(f1)


def save_checkpoint(model, optimizer, scheduler, epoch, val_metric, extra_info, path):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'epoch': epoch,
        'val_metric': val_metric,
        'architecture': 'efficientnet_b0',
        'class_mapping': {'REAL': 0, 'DEEPFAKE': 1},
        'threshold': 0.4,
        'input_size': IMAGE_SIZE,
        **extra_info
    }, path)

print('✅ Training functions defined!')

## Step 9: 🏋️ Stage 1 — Train Classifier Head (Frozen Backbone)

In [ ]:
model     = create_model(dropout=DROPOUT, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler    = GradScaler()
history   = {'train_loss': [], 'val_loss': [], 'train_acc': [],
             'val_acc': [], 'auc': [], 'f1': []}

# Freeze backbone
for p in model.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2)

S1_EPOCHS  = min(EPOCHS // 2, 10)
best_loss  = float('inf')
wait       = 0

print(f'\n{'='*60}')
print(f'STAGE 1: Training classifier head ({S1_EPOCHS} epochs)')
print(f'{'='*60}')

for epoch in range(1, S1_EPOCHS + 1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl, va, auc, f1 = validate(model, val_loader, criterion)
    scheduler.step(vl)
    elapsed = time.time() - t0

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    history['auc'].append(auc);       history['f1'].append(f1)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch:02d}/{S1_EPOCHS}  '
          f'train[loss={tl:.4f} acc={ta:.3f}]  '
          f'val[loss={vl:.4f} acc={va:.3f} AUC={auc:.3f} F1={f1:.3f}]  '
          f'lr={lr_now:.2e}  {elapsed:.0f}s')

    if vl < best_loss - 1e-4:
        best_loss = vl; wait = 0
        save_checkpoint(model, optimizer, scheduler, epoch, best_loss,
                       {'stage': 1, 'val_auc': auc, 'val_f1': f1}, CHECKPOINT)
        print(f'  ✅ Saved best → {CHECKPOINT}')
    else:
        wait += 1
        if wait >= 3:
            print('  ⏹️  Early stopping (Stage 1)'); break

print(f'\nStage 1 complete. Best val_loss = {best_loss:.4f}')

## Step 10: 🔓 Stage 2 — Fine-Tune Full Network

In [ ]:
# Reload Stage-1 best weights
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Reloaded Stage-1 best weights (epoch={ckpt["epoch"]})')

# Unfreeze ALL layers
for p in model.parameters():
    p.requires_grad = True

S2_EPOCHS = EPOCHS - S1_EPOCHS
optimizer2 = torch.optim.AdamW(model.parameters(), lr=LR/10, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=S2_EPOCHS, eta_min=1e-6)

best_loss2 = float('inf')
wait2 = 0

print(f'\n{"="*60}')
print(f'STAGE 2: Fine-tuning full network ({S2_EPOCHS} epochs)')
print(f'{"="*60}')

for epoch in range(1, S2_EPOCHS + 1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer2, scaler)
    vl, va, auc, f1 = validate(model, val_loader, criterion)
    scheduler2.step()
    elapsed = time.time() - t0

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    history['auc'].append(auc);       history['f1'].append(f1)

    lr_now = optimizer2.param_groups[0]['lr']
    print(f'Ep {epoch:02d}/{S2_EPOCHS}  '
          f'train[loss={tl:.4f} acc={ta:.3f}]  '
          f'val[loss={vl:.4f} acc={va:.3f} AUC={auc:.3f} F1={f1:.3f}]  '
          f'lr={lr_now:.2e}  {elapsed:.0f}s')

    if vl < best_loss2 - 1e-4:
        best_loss2 = vl; wait2 = 0
        save_checkpoint(model, optimizer2, scheduler2, epoch, best_loss2,
                       {'stage': 2, 'val_auc': auc, 'val_f1': f1}, CHECKPOINT)
        print(f'  ✅ Saved best → {CHECKPOINT}')
    else:
        wait2 += 1
        if wait2 >= 5:
            print('  ⏹️  Early stopping (Stage 2)'); break

print(f'\nStage 2 complete. Best val_loss = {best_loss2:.4f}')

## Step 11: 📊 Training Curves & Final Evaluation

In [ ]:
# Load best model for evaluation
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model: epoch={ckpt["epoch"]}, val_loss={ckpt["val_metric"]:.4f}')
print(f'val_AUC={ckpt.get("val_auc", 0):.4f}, val_F1={ckpt.get("val_f1", 0):.4f}')

# ── Training Curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(history['train_loss']) + 1)

# Add stage separator
stage_sep = S1_EPOCHS

for ax, (key_train, key_val, title, ylabel) in zip(
    axes.ravel(),
    [('train_loss', 'val_loss', 'Loss', 'Loss'),
     ('train_acc',  'val_acc',  'Accuracy', 'Accuracy'),
     ('auc', None, 'ROC-AUC (Validation)', 'AUC'),
     ('f1',  None, 'F1-Score (Validation)', 'F1')]):

    ax.plot(epochs_range, history[key_train], 'b-o', markersize=4, label='Train' if key_val else 'Val')
    if key_val:
        ax.plot(epochs_range, history[key_val], 'r-o', markersize=4, label='Val')
    ax.axvline(stage_sep + 0.5, color='gray', linestyle='--', alpha=0.5, label='Stage boundary')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training curves to Drive!')

# ── Final Classification Report ────────────────────────────────────────────
print('\n' + '='*60)
print('FINAL EVALUATION on Validation Set')
print('='*60)

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labs in tqdm(val_loader, desc='Evaluating'):
        with autocast():
            logits = model(imgs.to(device, non_blocking=True)).squeeze(1)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labs.numpy())

# Evaluate at multiple thresholds
print('\nThreshold Analysis:')
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5]:
    preds = (np.array(all_probs) >= thresh).astype(int)
    f1  = f1_score(all_labels, preds, zero_division=0)
    acc = (preds == np.array(all_labels)).mean()
    print(f'  thresh={thresh:.2f}  acc={acc:.4f}  f1={f1:.4f}')

print('\nAt threshold=0.4:')
final_preds = (np.array(all_probs) >= 0.4).astype(int)
print(classification_report(all_labels, final_preds,
                            target_names=['REAL', 'FAKE'], zero_division=0))

try:
    auc = roc_auc_score(all_labels, all_probs)
    print(f'ROC-AUC: {auc:.4f}')
except:
    pass

# Confusion Matrix
cm = confusion_matrix(all_labels, final_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (threshold=0.4)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: 🧪 Test on Sample Images (Qualitative)

In [ ]:
import glob
from IPython.display import display

@torch.no_grad()
def predict_image(img_path, threshold=0.4):
    img = Image.open(img_path).convert('RGB')
    tensor = val_transform(img).unsqueeze(0).to(device)
    with autocast():
        logit = model(tensor).squeeze()
    prob_fake = torch.sigmoid(logit).item()
    label = 'DEEPFAKE 🔴' if prob_fake >= threshold else 'REAL 🟢'
    return label, prob_fake, img

# Find some test images automatically
test_images = []
if VAL_ROOT:
    for cls in ['real', 'fake']:
        cls_dir = os.path.join(VAL_ROOT, cls)
        if os.path.isdir(cls_dir):
            imgs = sorted(glob.glob(f'{cls_dir}/*.jpg') + glob.glob(f'{cls_dir}/*.png'))[:3]
            test_images.extend([(p, cls) for p in imgs])

if not test_images:
    print('No test images found. Upload some images manually and set test_images list.')
else:
    n = min(len(test_images), 6)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.ravel() if rows > 1 else [axes] if cols == 1 else axes.ravel()

    for i, (img_path, true_label) in enumerate(test_images[:n]):
        pred_label, prob_fake, img = predict_image(img_path)
        color = 'red' if 'DEEPFAKE' in pred_label else 'green'

        ax = axes[i] if n > 1 else axes
        ax.imshow(img)
        ax.set_title(
            f'True: {true_label.upper()}\n{pred_label}\nFake={prob_fake*100:.1f}%',
            color=color, fontsize=11, fontweight='bold'
        )
        ax.axis('off')

    # Hide unused subplots
    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved sample predictions to Drive!')

## Step 13: 💾 Download Model to Your Local Machine

In [ ]:
# The model is already saved to Google Drive at:
print(f'✅ Model saved to Google Drive: {CHECKPOINT}')
print(f'   File size: {os.path.getsize(CHECKPOINT) / 1e6:.1f} MB')

# Option 1: Download directly to your computer
from google.colab import files
print('\nDownloading model to your computer...')
files.download(CHECKPOINT)

# After downloading, copy the file to:
# E:\deepfake ai detection\deepfake-detection\models\best_model.pth

## Step 14: 🔄 (Optional) Add More Datasets from Awesome-Deepfakes

### Supported Datasets from https://github.com/Daisy-Zhang/Awesome-Deepfakes

| Dataset | Type | Size | Access |
|---------|------|------|--------|
| **FaceForensics++** | Face swaps | ~1.5M frames | Request form |
| **Celeb-DF v2** | Celebrity fakes | 590 videos | [GitHub](https://github.com/yuezunli/celeb-deepfakeforensics) |
| **DFDC** | Facebook challenge | 100k videos | [Kaggle](https://www.kaggle.com/c/deepfake-detection-challenge) |
| **WildDeepfake** | In-the-wild | 7,314 clips | Request form |
| **FaceShifter** | High-fidelity swaps | — | Research use |

### How to add more data:

In [ ]:
# ── Combine multiple datasets ───────────────────────────────────────────────
# If you have multiple dataset folders, combine them:

# Example: Add Celeb-DF v2 frames
# EXTRA_TRAIN_ROOTS = [
#     '/content/dataset/celebdf/train',
#     '/content/dataset/faceforensics/train',
# ]

# class CombinedDataset(Dataset):
#     def __init__(self, roots, transform=None):
#         self.transform = transform
#         self.samples = []
#         for root in roots:
#             ds = DeepfakeDataset(root)
#             self.samples.extend(ds.samples)
#         print(f'Combined: {len(self.samples)} samples from {len(roots)} datasets')
#
#     def __len__(self): return len(self.samples)
#     def __getitem__(self, idx):
#         path, label = self.samples[idx]
#         img = Image.open(path).convert('RGB')
#         if self.transform: img = self.transform(img)
#         return img, torch.tensor(label, dtype=torch.float32)

print('Uncomment and modify the code above to add more datasets!')
print('Your model architecture supports any number of additional training images.')

## ✅ Summary

| Step | What Happened |
|------|---------------|
| Dataset | Loaded with class-imbalance handling |
| Stage 1 | Trained classifier head (frozen backbone) |
| Stage 2 | Fine-tuned entire network |
| Model | Saved to Google Drive |

### 📥 Next Steps — Install on Local Machine:
1. Download `deepfake_best_model.pth` from Drive
2. Copy it to: `E:\deepfake ai detection\deepfake-detection\models\best_model.pth`
3. Start the app: `uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload`
4. Open browser: `http://localhost:8000`